In [26]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import AllChem
from rdkit.Geometry import Point3D
import py3Dmol
IPythonConsole.ipython_3d = True

In [27]:
def draw3D(mol):
    mb = Chem.MolToMolBlock(mol)
    p = py3Dmol.view(width=400, height=400)
    p.addModel(mb, 'mol')
    p.setStyle({'stick': {}, 'sphere': {'scale': 0.3}})
    p.zoomTo()
    return p.show()

In [28]:
mol = Chem.MolFromSmiles('CC.CC')
mol = Chem.AddHs(mol)

coord_map = {
    1: Point3D(0.0, 0.0, 0.0),
    2: Point3D(10.0, 0.0, 0.0),
}

params = AllChem.ETKDGv3()
params.useSmallRingTorsions = True
params.useMacrocycle14config = True
params.clearConfs = False
params.pruneRmsThresh = 0.5
params.embedFragmentsSeparately = False
params.SetCoordMap(coord_map)

conf_ids = AllChem.EmbedMultipleConfs(
    mol,
    numConfs=5,
    params=params
)

draw3D(mol)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [29]:
mol = Chem.MolFromSmiles('C[NH+](C)C1=C([BH2-]C2=CC=CN2C)C=CC=C1')
mol = Chem.MolFromSmiles('CN1C=CC=C1[BH2-]C2=C([NH+]3C(C)(C)CCCC3(C)C)C=CC=C2')
mol = Chem.AddHs(mol)

query = Chem.MolFromSmarts('[#1]~[#7]~[*]~[*]~[#5]~[#6]')

matches = mol.GetSubstructMatches(query)
print(matches)

H = matches[0][0]
N = matches[0][1]
B = matches[0][4]
C = matches[0][5]

coord_map = {
    H: Point3D(-1.55862400, 0.10360000, 1.04789500),
    B: Point3D(0.37331800, -0.29150300, 1.70001600),
    N: Point3D(-2.41614400, -1.10144500, 0.73040300),
    C: Point3D(-0.65942900, 0.98644000, 1.32824300),
}

AllChem.EmbedMultipleConfs(mol,1,
        maxAttempts=0,
        randomSeed=0xF00D,
        useRandomCoords=False,
        pruneRmsThresh=0.5,
        coordMap=coord_map,
        ignoreSmoothingFailures=True,
        enforceChirality=True,
        useSmallRingTorsions=True)

draw3D(mol)

((31, 9, 8, 7, 6, 5),)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [30]:
mol = Chem.MolFromSmiles('CCCBr.[Br-]')
mol = Chem.AddHs(mol)
query = Chem.MolFromSmarts('[Br-].[#6]([#1])([#1])-[Br]')
matches = mol.GetSubstructMatches(query)
print(matches)

Brm = matches[0][0]
C = matches[0][1]
H1 = matches[0][2]
H2 = matches[0][3]
Br = matches[0][4]

coord_map = {
    C:   Point3D(0.000000,  0.000000,  0.000000),
    H1:  Point3D(0.537218,  0.930488,  0.000000),
    H2:  Point3D(0.537218, -0.930488,  0.000000),
    Brm: Point3D(0.000000,  0.000000, -2.514522),
    Br:  Point3D(0.000000,  0.000000,  2.514522),
}

params = AllChem.ETKDGv3()
params.useSmallRingTorsions = True
params.useMacrocycle14config = True
params.clearConfs = False
params.pruneRmsThresh = 0.5
params.embedFragmentsSeparately = False
params.SetCoordMap(coord_map)

conf_ids = AllChem.EmbedMultipleConfs(
    mol,
    numConfs=5,
    params=params
)

draw3D(mol)

((4, 2, 10, 11, 3),)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [31]:
#based on racerTS
#racerTS also has similar ASE plugin so one can use other energy functions
from rdkit import Chem
from rdkit.Chem import AllChem


def minimize_structure(
    mol,
    coord_map=None,
    method="UFF",
    force_constant=1e6,
    max_iters=500,
    conf_ids=None,
    verbose=True,
):
    """
    Minimize conformers in `mol`.

    Parameters
    ----------
    mol : rdkit.Chem.Mol
        Molecule from the previous cell. It should already have conformers.
    coord_map : dict[int, rdkit.Geometry.Point3D]
        Atom indices and target coordinates to restrain during minimization.
        Use the same coord_map from the previous cell.
    method : str
        "UFF" or "MMFF".
    force_constant : float
        Strength of positional restraints.
    max_iters : int
        Maximum minimization iterations.
    conf_ids : list[int] or None
        Conformer IDs to minimize. If None, all conformers are minimized.
    verbose : bool
        Print minimization results.

    Returns
    -------
    mol : rdkit.Chem.Mol
        The minimized molecule.
    """

    if coord_map is None:
        coord_map = {}

    if mol.GetNumConformers() == 0:
        raise ValueError("mol has no conformers. Run EmbedMolecule or EmbedMultipleConfs first.")

    method = method.upper()

    if conf_ids is None:
        conf_ids = [conf.GetId() for conf in mol.GetConformers()]

    mmff_props = None
    if method == "MMFF":
        mmff_props = AllChem.MMFFGetMoleculeProperties(mol)
        if mmff_props is None:
            raise ValueError("MMFF parameters are not available for this molecule.")

    results = []

    for conf_id in conf_ids:
        if method == "UFF":
            ff = AllChem.UFFGetMoleculeForceField(
                mol,
                confId=conf_id,
                ignoreInterfragInteractions=False,
            )
        elif method == "MMFF":
            ff = AllChem.MMFFGetMoleculeForceField(
                mol,
                mmff_props,
                confId=conf_id,
                ignoreInterfragInteractions=False,
            )
        else:
            raise ValueError("method must be 'UFF' or 'MMFF'.")

        # Restrain atoms in coord_map to their target positions
        for atom_idx, point in coord_map.items():
            extra_point_idx = ff.AddExtraPoint(
                point.x,
                point.y,
                point.z,
                fixed=True,
            ) - 1

            ff.AddDistanceConstraint(
                extra_point_idx,
                atom_idx,
                0.0,
                0.0,
                force_constant,
            )

        ff.Initialize()
        not_converged = ff.Minimize(maxIts=max_iters)
        energy = ff.CalcEnergy()

        mol.GetConformer(conf_id).SetDoubleProp("energy", energy)
        results.append((conf_id, not_converged, energy))

    if verbose:
        for conf_id, not_converged, energy in results:
            status = "not converged" if not_converged else "converged"
            print(f"conf {conf_id}: {status}, energy = {energy:.6f}")

    return mol

In [32]:
mol = minimize_structure(
    mol,
    coord_map=coord_map,
    method="UFF",      # or "MMFF"
    force_constant=1e6,
    max_iters=500,
)

draw3D(mol)

conf 0: converged, energy = 163.789894
conf 1: converged, energy = 153.874662


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [33]:
mol.GetNumConformers()

2